In [2]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.optim import SGD
from torch.nn import SmoothL1Loss
device = 'cuda' if torch.cuda.is_available() else 'cpu'
df = pd.read_csv('solarpowergeneration.csv')
data = df.to_numpy(dtype=float)
data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)
X_np = data[:, 0:20]
Y_np = data[:, 20]
x_scaler = StandardScaler()
X_scaled = x_scaler.fit_transform(X_np)
y_scaler = StandardScaler()
Y_scaled = y_scaler.fit_transform(Y_np.reshape(-1, 1)).flatten()
X = torch.tensor(X_scaled, dtype=torch.float32).to(device)
Y = torch.tensor(Y_scaled, dtype=torch.float32).to(device)
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
class weather_net_v1(nn.Module):
    def __init__(self, in_feature, hidden_unit):
        super().__init__()
        self.layer1 = nn.Linear(in_feature, hidden_unit)
        self.layer2 = nn.Linear(hidden_unit, 2 * hidden_unit)
        self.layer3 = nn.Linear(2 * hidden_unit, 2 * hidden_unit)
        self.layer4 = nn.Linear(2 * hidden_unit, hidden_unit)
        self.layer5 = nn.Linear(hidden_unit, 1)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return torch.clamp(self.layer5(x), min=-5, max=5)  # Output bounded

# Initialize model
model_1 = weather_net_v1(X.shape[1], 64).to(device)

# Loss and optimizer
loss_fn = SmoothL1Loss()
optimizer = SGD(model_1.parameters(), lr=0.001, momentum=0.9)

# Training loop
for epoch in range(5):
    model_1.train()
    for i in range(len(x_train)):
        optimizer.zero_grad()
        y_pred = model_1(x_train[i])
        loss = loss_fn(y_pred, y_train[i].unsqueeze(0))
        if torch.isnan(loss):
            print(f"NAN DETECTED at epoch {epoch}, sample {i}")
            print("Input:", x_train[i])
            print("Target:", y_train[i])
            exit()
        loss.backward()
        optimizer.step()

    # Evaluation
    model_1.eval()
    loss_avg = 0
    with torch.inference_mode():
        for i in range(len(x_test)):
            y_pred = model_1(x_test[i].unsqueeze(0))
            loss_avg += loss_fn(y_pred.squeeze(), y_test[i])
    print(f"Epoch {epoch+1}, Test Loss: {loss_avg / len(x_test):.4f}")

# Example: denormalized prediction
model_1.eval()
with torch.inference_mode():
    sample_input = x_test[0].unsqueeze(0)
    normalized_output = model_1(sample_input).cpu().numpy()
    predicted_real_value = y_scaler.inverse_transform(normalized_output.reshape(-1, 1))
    print(f"Predicted Solar Power (kWh): {predicted_real_value[0][0]:.2f}")


KeyboardInterrupt: 